# The Beta Distribution

In this notebook we shall sample a toy model of the 'Beta Distribution'.
For certain coefficient combinations, i.e. (0.5, 0.5), the beta distribution resembles a U-shape with asymptotic behaviour at 0 and 1 where p tends to infinity. Nevertheless the integral remains equal to 1. This structure presents a challenge for MCMC samplers due to the huge range of scale lengths required to sample in the interval 0 to 1 as well as very close to 0 and very close to 1.

Here we show that `essrs` is easily able to sample within these asymptotic regions.

In [ ]:
import numpy as np
from scipy.stats import beta
import matplotlib.pyplot as plt
import time

import essrs

#### Function Parameterisation

In [ ]:
ALPHA: float = 0.5
BETA: float = 0.5

#### MCMC Parameterisation

In [ ]:
N_SAMPLES: int = 1000
N_WALKERS: int = 1000
N_PARAMETERS: int = 1
N_BURN_IN: int = 1000
N_THREADS: int = 1
N_THINNING_PER_SAMPLE: int = None

In [ ]:
burn_move = essrs.moves.hybrid_ddc_move(one_in_n=10)
sampler_move = essrs.moves.differential_move()
config = essrs.EnsembleConfig(
    n_samples=N_SAMPLES,
    n_walkers=N_WALKERS,
    n_parameters=N_PARAMETERS,
    n_burn_in=N_BURN_IN,
    n_threads=N_THREADS,
    n_thinning_per_sample=N_THINNING_PER_SAMPLE
)

#### Initial Conditions

In [ ]:
ics = np.random.uniform(low=0.5, high=1, size=(N_WALKERS, N_PARAMETERS))

#### Sampling

In [ ]:
model = essrs.model.toy.beta_distribution(alpha=ALPHA, beta=BETA)

start = time.perf_counter()
fit_dyn, ll = essrs.run_mcmc(
    log_likelihood=model,
    config=config,
    burn_in_move=burn_move,
    sampling_move=sampler_move,
    initial_conditions=ics
)
elapsed = time.perf_counter() - start
print(f"Fit :: {elapsed:.3f} seconds")

In [ ]:
x = np.linspace(0, 1, 1000)
y = beta.pdf(x, ALPHA, BETA)

plt.figure()
plt.hist(fit_dyn.ravel(), bins=1000, density=True, histtype="step", color="b")
plt.plot(x, y, color="r", ls="-.")

plt.xlabel(r"$x$")

plt.ylabel(r"$P\left(x\right)$")
plt.show()